# DataFrame to LaTeX Table Converter

This notebook provides a dedicated tool (`LatexTableFormatter`) to format DataFrames into publication-ready LaTeX tables, with specialized support for:
- Concatenating central values and uncertainties into LaTeX `$val \pm unc$` syntax.
- Automatic uncertainty-based rounding (PDG standard: 2 significant figures for uncertainty with aligned central value decimal precision).
- Fixed decimal precision rounding.
- Auto-escaping LaTeX underscores and customizable tabular wrappers (`caption`, `label`, `column_format`, `resizebox`).

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from typing import Optional
from src.core.Validation.LatexTableFormatter import LatexTableFormatter

# Functions

In [5]:
def concatenate_pm_columns(
    df: pd.DataFrame,
    val_col: str,
    unc_col: str,
    new_col: Optional[str] = None,
    precision: Optional[int] = None,
    sig_figs: int = 2,
    drop_original: bool = False,
) -> pd.DataFrame:
    r"""Combines two columns into one formatted as '$val \pm unc$'."""
    return LatexTableFormatter.concatenate_columns(
        df=df,
        val_col=val_col,
        unc_col=unc_col,
        new_col=new_col,
        precision=precision,
        sig_figs=sig_figs,
        drop_original=drop_original,
    )


def dataframe_to_latex(
    df: pd.DataFrame,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    column_format: Optional[str] = None,
    escape_underscores: bool = True,
    resize_box: bool = False,
    index: bool = False,
) -> str:
    """Converts a DataFrame directly into a LaTeX table snippet."""
    return LatexTableFormatter.to_latex(
        df=df,
        caption=caption,
        label=label,
        column_format=column_format,
        escape_underscores=escape_underscores,
        resize_box=resize_box,
        index=index,
    )


TABLE_PRESETS = {
    "quadrant_strategy_metrics": {
        "pairs": {
            "PD": ("PD", "PD_unc"),
            "PF": ("PF", "PF_unc"),
            "SP": ("SP", "SP_unc"),
        },
        "keep_cols": ["Strategy", "PD", "PD_unc", "PF", "PF_unc", "SP", "SP_unc"],
    },
    "variable_js_divergence": {
        "pairs": {
            "JS Distance ($\\sqrt{D_{\\text{JS}}}$)": ("JS Divergence", "JS Uncertainty"),
        },
        "keep_cols": ["Feature", "Strategy", "JS Divergence", "JS Uncertainty"],
    },
    "multi_model_mean_summary": {
        "pairs": {
            "PD (Signal Eff)": ("PD mean [%]", "PD std [%]"),
            "PF (Fake Rate)": ("PF mean [%]", "PF std [%]"),
            "SP": ("SP mean [%]", "SP std [%]"),
        },
        "keep_cols": ["method", "PD mean [%]", "PD std [%]", "PF mean [%]", "PF std [%]", "SP mean [%]", "SP std [%]"],
    },
}


def discover_tables(project_root: Path = PROJECT_ROOT) -> pd.DataFrame:
    """Discovers saved CSV and Parquet tables under Plots/*/Tables and notebooks/Tables."""
    candidates = sorted(
        list((project_root / "Plots").glob("*/Tables/*.csv"))
        + list((project_root / "notebooks" / "Tables").glob("*.csv"))
        + list((project_root / "notebooks" / "Tables").glob("*.parquet"))
    )
    rows = [
        {
            "Index": i,
            "Folder": p.parent.parent.name if p.parent.name == "Tables" else p.parent.name,
            "File": p.name,
            "Path": str(p),
        }
        for i, p in enumerate(candidates)
    ]
    return pd.DataFrame(rows)


def load_table_df(table_path: Path) -> pd.DataFrame:
    """Loads a CSV or Parquet table into a DataFrame."""
    table_path = Path(table_path)
    if table_path.suffix.lower() == ".parquet":
        return pd.read_parquet(table_path)
    return pd.read_csv(table_path)


def resolve_latex_output_path(table_path: Path, custom_output_dir: Optional[Path] = None) -> Path:
    """Builds the standardized LaTeX output path: <run_dir>/LaTeX/<table_stem>_latex_table.tex."""
    table_path = Path(table_path)
    if custom_output_dir is not None:
        out_dir = Path(custom_output_dir)
    elif table_path.parent.name == "Tables":
        out_dir = table_path.parent.parent / "LaTeX"
    else:
        out_dir = table_path.parent / "LaTeX"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"{table_path.stem}_latex_table.tex"


def format_and_save_latex_table(
    table_path: Path,
    metric_pairs: Optional[dict] = None,
    keep_cols: Optional[list[str]] = None,
    precision: Optional[int] = None,
    sig_figs: int = 2,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    column_format: Optional[str] = None,
    resize_box: bool = False,
    custom_output_dir: Optional[Path] = None,
    max_rows: Optional[int] = None,
) -> tuple[pd.DataFrame, str, Path]:
    """Loads a table, applies preset or custom uncertainty formatting, and saves <stem>_latex_table.tex."""
    table_path = Path(table_path)
    df = load_table_df(table_path)
    if max_rows is not None:
        df = df.head(max_rows)

    matched_preset = next((v for k, v in TABLE_PRESETS.items() if k in table_path.stem), None)
    active_pairs = metric_pairs if metric_pairs is not None else (matched_preset["pairs"] if matched_preset else {})
    active_cols = keep_cols if keep_cols is not None else (matched_preset["keep_cols"] if matched_preset else None)

    if active_cols is not None:
        valid_cols = [c for c in active_cols if c in df.columns]
        if valid_cols:
            df = df[valid_cols]

    valid_pairs = {
        new_col: (v_col, u_col)
        for new_col, (v_col, u_col) in active_pairs.items()
        if v_col in df.columns and u_col in df.columns
    }
    if valid_pairs:
        formatted_df = LatexTableFormatter.merge_uncertainty_pairs(
            df,
            pairs=valid_pairs,
            precision=precision,
            sig_figs=sig_figs,
            drop_original=True,
        )
    else:
        formatted_df = df.copy()

    latex_str = dataframe_to_latex(
        formatted_df,
        caption=caption,
        label=label or f"tab:{table_path.stem}",
        column_format=column_format,
        resize_box=resize_box,
    )
    tex_path = resolve_latex_output_path(table_path, custom_output_dir=custom_output_dir)
    tex_path.write_text(latex_str, encoding="utf-8")
    return formatted_df, latex_str, tex_path

# Exemples

## Example A: Synthetic Distance Metrics & Automatic Rounding (PDG Style)


In [4]:
# Create sample DataFrame with diverse magnitudes
sample_df = pd.DataFrame({
    "Feature": ["trig_L2_calo_et", "trig_L2_calo_eta", "Eratio"],
    "Comparison": ["Model V9 vs Model V10", "Model V9 vs Model V10", "Model V9 vs Model V10"],
    "KS_Distance": [0.008336, 0.006995, 0.233209],
    "KS_Uncertainty": [0.005644, 0.006027, 0.008621],
    "Wasserstein": [16.628878, 0.010359, 0.155185],
    "Wasserstein_Uncertainty": [23.797069, 0.012521, 0.005474],
    "JS_Divergence": [0.000370, 0.000194, 0.048428],
    "JS_Uncertainty": [0.000782, 0.000495, 0.003412],
})

display(sample_df)

,Feature,Comparison,KS_Distance,KS_Uncertainty,Wasserstein,Wasserstein_Uncertainty,JS_Divergence,JS_Uncertainty
0,trig_L2_calo_et,Model V9 vs Model V10,0.008336,0.005644,16.628878,23.797069,0.000370,0.000782
1,trig_L2_calo_eta,Model V9 vs Model V10,0.006995,0.006027,0.010359,0.012521,0.000194,0.000495
2,Eratio,Model V9 vs Model V10,0.233209,0.008621,0.155185,0.005474,0.048428,0.003412


## Example B: Merging Multiple Columns with Auto-Rounding (`sig_figs=2`)

In [5]:
# Merge pairs into unified LaTeX columns
pairs = {
    "KS Distance ($D$)": ("KS_Distance", "KS_Uncertainty"),
    "Wasserstein ($W_1$)": ("Wasserstein", "Wasserstein_Uncertainty"),
    "JS Divergence ($D_{\\text{JS}}$)": ("JS_Divergence", "JS_Uncertainty"),
}

# Automatic PDG rounding
formatted_auto = LatexTableFormatter.merge_uncertainty_pairs(
    sample_df,
    pairs=pairs,
    precision=None,
    sig_figs=2,
    drop_original=True,
)

print("=== Preview in Notebook ===")
display(formatted_auto)

print("=== Generated LaTeX Code ===")
latex_code = dataframe_to_latex(
    formatted_auto,
    caption="Statistical Distance Metrics with Automatic Uncertainty Rounding.",
    label="tab:distance_metrics",
)
print(latex_code)

=== Preview in Notebook ===


,Feature,Comparison,KS Distance ($D$),Wasserstein ($W_1$),JS Divergence ($D_{\text{JS}}$)
0,trig_L2_calo_et,Model V9 vs Model V10,$0.0083 \pm 0.0056$,$17 \pm 24$,$0.00037 \pm 0.00078$
1,trig_L2_calo_eta,Model V9 vs Model V10,$0.0070 \pm 0.0060$,$0.010 \pm 0.013$,$0.00019 \pm 0.00049$
2,Eratio,Model V9 vs Model V10,$0.2332 \pm 0.0086$,$0.1552 \pm 0.0055$,$0.0484 \pm 0.0034$


=== Generated LaTeX Code ===
\begin{table}[htbp]
\centering
\caption{Statistical Distance Metrics with Automatic Uncertainty Rounding.}
\label{tab:distance_metrics}
\begin{tabular}{l l l l l}
\toprule
Feature & Comparison & KS Distance ($D$) & Wasserstein ($W_1$) & JS Divergence ($D_{\text{JS}}$) \\
\midrule
trig\_L2\_calo\_et & Model V9 vs Model V10 & $0.0083 \pm 0.0056$ & $17 \pm 24$ & $0.00037 \pm 0.00078$ \\
trig\_L2\_calo\_eta & Model V9 vs Model V10 & $0.0070 \pm 0.0060$ & $0.010 \pm 0.013$ & $0.00019 \pm 0.00049$ \\
Eratio & Model V9 vs Model V10 & $0.2332 \pm 0.0086$ & $0.1552 \pm 0.0055$ & $0.0484 \pm 0.0034$ \\
\bottomrule
\end{tabular}
\end{table}



## Example C: Fixed Decimal Precision Rounding (`precision=4`)

In [6]:
# Merge with fixed precision (4 decimal places)
formatted_fixed = LatexTableFormatter.merge_uncertainty_pairs(
    sample_df,
    pairs=pairs,
    precision=4,
    drop_original=True,
)

display(formatted_fixed)
print(dataframe_to_latex(formatted_fixed))

,Feature,Comparison,KS Distance ($D$),Wasserstein ($W_1$),JS Divergence ($D_{\text{JS}}$)
0,trig_L2_calo_et,Model V9 vs Model V10,$0.0083 \pm 0.0056$,$16.6289 \pm 23.7971$,$0.0004 \pm 0.0008$
1,trig_L2_calo_eta,Model V9 vs Model V10,$0.0070 \pm 0.0060$,$0.0104 \pm 0.0125$,$0.0002 \pm 0.0005$
2,Eratio,Model V9 vs Model V10,$0.2332 \pm 0.0086$,$0.1552 \pm 0.0055$,$0.0484 \pm 0.0034$


\begin{tabular}{l l l l l}
\toprule
Feature & Comparison & KS Distance ($D$) & Wasserstein ($W_1$) & JS Divergence ($D_{\text{JS}}$) \\
\midrule
trig\_L2\_calo\_et & Model V9 vs Model V10 & $0.0083 \pm 0.0056$ & $16.6289 \pm 23.7971$ & $0.0004 \pm 0.0008$ \\
trig\_L2\_calo\_eta & Model V9 vs Model V10 & $0.0070 \pm 0.0060$ & $0.0104 \pm 0.0125$ & $0.0002 \pm 0.0005$ \\
Eratio & Model V9 vs Model V10 & $0.2332 \pm 0.0086$ & $0.1552 \pm 0.0055$ & $0.0484 \pm 0.0034$ \\
\bottomrule
\end{tabular}



# Load, Format & Save Tables

In [ ]:
# 1. Discover all available tables under Plots/*/Tables and notebooks/Tables
tables_catalog = discover_tables(PROJECT_ROOT)
display(tables_catalog)

In [ ]:
# 2. Select a table by Index from `tables_catalog` OR specify `CUSTOM_TABLE_PATH` directly
TABLE_INDEX: int = 0
CUSTOM_TABLE_PATH: Optional[Path] = None

PRECISION: Optional[int] = None   # None = automatic PDG rounding, or int (e.g., 3)
SIG_FIGS: int = 2
CAPTION: Optional[str] = None
LABEL: Optional[str] = None       # None = auto tab:<file_stem>
MAX_ROWS: Optional[int] = None

selected_path = Path(CUSTOM_TABLE_PATH) if CUSTOM_TABLE_PATH is not None else Path(tables_catalog.loc[TABLE_INDEX, "Path"])
formatted_df, latex_str, saved_tex_path = format_and_save_latex_table(
    table_path=selected_path,
    precision=PRECISION,
    sig_figs=SIG_FIGS,
    caption=CAPTION,
    label=LABEL,
    max_rows=MAX_ROWS,
)

print(f"Source table: {selected_path}")
print(f"Saved LaTeX:  {saved_tex_path}")
display(formatted_df)
print("\n--- Exported LaTeX ---")
print(latex_str)